# TP1 — Optimisation continue et linéaire

**Séance 1 — Optimisation pour l'IA — MS IA de Confiance, CentraleSupélec**

Dans ce TP vous allez, **étape par étape** :

1. Implémenter la **descente de gradient** à partir de zéro et étudier l'effet du pas.
2. Implémenter et comparer une méthode **accélérée (momentum / Nesterov)**.
3. Résoudre un **problème continu contraint** avec une méthode de barrière, `scipy.optimize`, et `cvxpy`.
4. Formuler et résoudre un **programme linéaire** avec `scipy.optimize.linprog` et `PuLP`, et le comparer à votre résolution du simplexe à la main faite en TD.
5. Démarrer votre **projet individuel ExedBike (Partie 1)**.

**Comment ce notebook est organisé.** Chaque exercice est découpé en petites étapes : une cellule **`# A COMPLETER`** à écrire, suivie presque toujours d'une cellule **de vérification** (marquée ✅) qui teste automatiquement votre code et vous dit si c'est correct — lancez les cellules **dans l'ordre**, une par une, et ne passez à la suivante que lorsque la vérification est au vert. Aucune solution n'est donnée à l'avance : le but est que chaque ligne que vous exécutez, vous l'ayez écrite (ou complétée) vous-même.

Bibliothèques nécessaires : `numpy`, `matplotlib`, `autograd`, `scipy`, `cvxpy`, `pulp`.

> **Installation / choix du kernel.** La cellule suivante installe automatiquement les paquets manquants dans le kernel Python actuellement sélectionné. Si vous obtenez malgré tout `ModuleNotFoundError` : vérifiez le **kernel** utilisé par ce notebook (dans VS Code : en haut à droite du notebook, bouton "Select Kernel" ; dans Jupyter : menu Kernel > Change kernel). Un `ModuleNotFoundError` sur `numpy` en particulier signifie presque toujours que le kernel sélectionné pointe vers un interpréteur Python différent de celui où les paquets sont installés (ou vers un interpréteur « nu » sans aucun paquet) — sélectionnez un autre kernel, ou relancez la cellule ci-dessous puis **redémarrez le kernel** (Restart) avant de continuer.

In [ ]:
import sys, subprocess, importlib

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"[setup] installation de {pkg} dans {sys.executable} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for _pkg, _imp in [
    ("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
    ("autograd", "autograd"), ("cvxpy", "cvxpy"), ("pulp", "pulp"),
]:
    _ensure(_pkg, _imp)

import numpy as np
import matplotlib.pyplot as plt
from autograd import grad
import autograd.numpy as anp

np.random.seed(42)
plt.rcParams["figure.figsize"] = (6, 4)
print("[setup] environnement pret :", sys.executable)


## Exercice 1 — Descente de gradient à partir de zéro

Rappel de la règle de mise à jour :
$$ \mathbf{w}_{k+1} = \mathbf{w}_k - \alpha \nabla J(\mathbf{w}_k). $$

**1.1** Complétez `gradient_descent` ci-dessous : à chaque itération, calculez le gradient de `cost_fun` au point courant (utilisez `g`, déjà construit avec `autograd`), arrêtez-vous si sa norme passe sous `eps`, sinon mettez à jour `w` et ajoutez le nouveau point à `history`.

In [ ]:
def gradient_descent(cost_fun, w_init, alpha, epochs, eps=1e-12):
    '''Minimise cost_fun par descente de gradient.

    ENTREES
    cost_fun : callable, fonction de coût J(w)
    w_init   : array-like, point initial
    alpha    : float, pas (learning rate)
    epochs   : int, nombre maximal d'itérations
    eps      : float, arrêt si la norme du gradient passe sous eps

    SORTIES
    w       : point final (np.array)
    history : liste des itérés successifs (y compris w_init)
    '''
    w = np.array(w_init, dtype=float)
    history = [w.copy()]
    g = grad(cost_fun)   # g(w) renvoie le gradient de cost_fun en w (autograd)

    for k in range(epochs):
        # A COMPLETER (3-4 lignes) :
        # 1) calculez gk = g(w) (convertissez en np.array)
        # 2) si la norme de gk est < eps : "break"
        # 3) sinon : mettez à jour w = w - alpha * gk
        # 4) ajoutez une copie de w à "history"
        raise NotImplementedError("Complétez gradient_descent")

    return w, history


**✅ Vérification 1.1** — testez `gradient_descent` sur un problème 1D trivial dont vous connaissez la solution exacte : $J(w) = (w-3)^2$, minimum en $w=3$.

In [ ]:
def _J1d(w):
    return (w[0] - 3.0) ** 2

_w_check, _hist_check = gradient_descent(_J1d, w_init=[0.0], alpha=0.1, epochs=500, eps=1e-12)
assert abs(_w_check[0] - 3.0) < 1e-3, f"attendu w*≈3.0, obtenu {_w_check[0]}"
assert len(_hist_check) > 1, "history doit contenir plusieurs points"
print(f"OK : convergé en {len(_hist_check)-1} itérations vers w* = {_w_check[0]:.5f}")


**1.2** Appliquez la descente de gradient au bol quadratique
$$ J(w_1, w_2) = 3(w_1 - 1)^2 + (w_2 + 2)^2 $$
en partant de $w_0 = (4, 3)$, pour $\alpha = 0.05$ **puis** $\alpha = 0.3$. La fonction `show_history` ci-dessous (déjà écrite) trace les lignes de niveau de $J$ et la trajectoire des itérés — complétez juste la boucle qui appelle `gradient_descent` pour chaque `alpha` et affiche le résultat.

In [ ]:
def J_quad(w):
    return 3 * (w[0] - 1) ** 2 + (w[1] + 2) ** 2


def show_history(history, cost_fun, xlim, ylim, title=""):
    a_values, b_values = np.meshgrid(
        np.linspace(*xlim), np.linspace(*ylim)
    )
    Z = np.array(
        [cost_fun([a, b]) for a, b in zip(a_values.flat, b_values.flat)]
    ).reshape(a_values.shape)
    plt.figure()
    plt.contour(a_values, b_values, Z, levels=20, cmap="viridis")
    h = np.array(history)
    plt.plot(h[:, 0], h[:, 1], "o-", color="crimson", markersize=3)
    plt.plot(h[0, 0], h[0, 1], "ks", label="départ")
    plt.plot(h[-1, 0], h[-1, 1], "k*", markersize=12, label="arrivée")
    plt.legend()
    plt.title(title)
    plt.xlabel("$w_1$")
    plt.ylabel("$w_2$")
    plt.show()


# A COMPLETER : pour alpha dans [0.05, 0.3], appelez gradient_descent(J_quad, w_init=[4.0, 3.0],
# alpha=alpha, epochs=60), puis show_history(hist, J_quad, xlim=(-1, 5), ylim=(-4, 4), title=...)
for alpha in [0.05, 0.3]:
    raise NotImplementedError("Complétez la boucle ci-dessus")


**Vos observations :** *(à rédiger ici avant de continuer)* pourquoi la trajectoire zigzague-t-elle pour l'un des deux `alpha` et pas pour l'autre ? Indice : regardez le conditionnement de la Hessienne de $J$ (le rapport entre ses deux coefficients quadratiques, $3$ et $1$).

## Exercice 2 — Méthodes accélérées : momentum & Nesterov

**2.1** Implémentez le **momentum (heavy-ball)** :
$$ \mathbf{v}_{k+1} = \beta \mathbf{v}_k - \alpha \nabla J(\mathbf{w}_k), \qquad \mathbf{w}_{k+1} = \mathbf{w}_k + \mathbf{v}_{k+1} $$
et le **gradient accéléré de Nesterov** :
$$ \mathbf{v}_{k+1} = \beta \mathbf{v}_k - \alpha \nabla J(\mathbf{w}_k + \beta \mathbf{v}_k), \qquad \mathbf{w}_{k+1} = \mathbf{w}_k + \mathbf{v}_{k+1} $$
Les deux se codent dans **une seule** fonction `momentum_gd` : le paramètre `nesterov` choisit si le gradient est évalué en `w` (momentum) ou au point "anticipé" `w + beta*v` (Nesterov).

In [ ]:
def momentum_gd(cost_fun, w_init, alpha, beta, epochs, eps=1e-12, nesterov=False):
    w = np.array(w_init, dtype=float)
    v = np.zeros_like(w)
    history = [w.copy()]
    g = grad(cost_fun)
    for k in range(epochs):
        # A COMPLETER (4-5 lignes) :
        # 1) lookahead = w + beta*v si nesterov, sinon w
        # 2) gk = g(lookahead)
        # 3) arrêt si ||gk|| < eps
        # 4) v = beta*v - alpha*gk ; w = w + v
        # 5) ajoutez une copie de w à "history"
        raise NotImplementedError("Complétez momentum_gd")
    return w, history


**✅ Vérification 2.1** — même test que pour l'Exercice 1 (doit converger vers $w=3$, avec et sans Nesterov).

In [ ]:
_w1, _h1 = momentum_gd(_J1d, w_init=[0.0], alpha=0.1, beta=0.5, epochs=500, eps=1e-10, nesterov=False)
_w2, _h2 = momentum_gd(_J1d, w_init=[0.0], alpha=0.1, beta=0.5, epochs=500, eps=1e-10, nesterov=True)
assert abs(_w1[0] - 3.0) < 1e-2, f"momentum : attendu w*≈3.0, obtenu {_w1[0]}"
assert abs(_w2[0] - 3.0) < 1e-2, f"nesterov : attendu w*≈3.0, obtenu {_w2[0]}"
print(f"OK : momentum -> w*={_w1[0]:.4f} en {len(_h1)-1} it. ; nesterov -> w*={_w2[0]:.4f} en {len(_h2)-1} it.")


**2.2** Comparez, sur le même quadratique mal conditionné $J\_quad$ ($\alpha = 0.15$), le nombre d'itérations nécessaires pour atteindre $\|\nabla J\| < 10^{-6}$ pour : descente de gradient simple, momentum ($\beta=0.8$), Nesterov ($\beta=0.8$). Complétez la boucle `for name, fn in [...]` (donnez à chaque `fn` un appel à `gradient_descent` ou `momentum_gd` avec les bons paramètres) ; le tracé du diagramme en barres est déjà écrit.

In [ ]:
results = {}
for name, fn in [
    ("Descente de gradient", None),  # A COMPLETER : lambda: gradient_descent(J_quad, [4.0, 3.0], 0.15, 500, eps=1e-6)
    ("Momentum", None),              # A COMPLETER : lambda: momentum_gd(J_quad, [4.0, 3.0], 0.15, 0.8, 500, eps=1e-6)
    ("Nesterov", None),              # A COMPLETER : lambda: momentum_gd(..., nesterov=True)
]:
    w_star, hist = fn()
    results[name] = len(hist) - 1
    print(f"{name:22s}: {len(hist)-1:4d} itérations -> w* = {np.round(w_star, 4)}")

plt.figure()
plt.bar(results.keys(), results.values(), color=["#5B7FB5", "#E3946A", "#8CB369"])
plt.ylabel("itérations pour converger")
plt.title("Effet de l'accélération (alpha=0.15)")
plt.show()


**Vos observations :** *(à rédiger ici)* le résultat vous surprend-il ? Le momentum est-il plus rapide que la descente de gradient simple **ici** ? Indice : le conditionnement de `J_quad` est modéré ($\kappa=3$) — essayez de refaire ce comparatif avec un `J_quad` bien plus mal conditionné (remplacez le `3` par `30` ou `300`) pour voir ce que ça change.

## Exercice 3 — Optimisation continue sous contraintes

Considérons
$$ \min_{x_1,x_2} \; (x_1 - 3/2)^2 + (x_2 - 1/2)^4 \quad \text{s.c.} \quad |x_1| + |x_2| \le 1 .$$

**3.1** Résolvez-le de **trois façons indépendantes**, une cellule à la fois, et vérifiez qu'elles s'accordent.

**(a) Barrière logarithmique.** L'idée : remplacer les 4 contraintes linéaires équivalentes à $|x_1|+|x_2|\le 1$ (à savoir $g_i(x)\ge 0$ pour $g_1=1-x_1-x_2$, $g_2=1+x_1+x_2$, $g_3=1-x_1+x_2$, $g_4=1+x_1-x_2$) par une pénalité $-\xi\sum_i \log g_i(x)$ ajoutée à l'objectif, et résoudre un problème **non contraint** avec `scipy.optimize.minimize` (méthode Nelder-Mead).

In [ ]:
import cvxpy as cp
from scipy.optimize import minimize

def f0(x1, x2):
    return (x1 - 1.5) ** 2 + (x2 - 0.5) ** 4

def f0_barrier(x, xi=0.05):
    x1, x2 = x
    # A COMPLETER : g = liste des 4 contraintes [1-x1-x2, 1+x1+x2, 1-x1+x2, 1+x1-x2]
    # si min(g) <= 0 : renvoyer une grande pénalité (ex. 1e6) -- on est hors du domaine faisable
    # sinon : renvoyer f0(x1, x2) - xi * somme(log(g_i))
    raise NotImplementedError("Complétez f0_barrier")

sol_barrier = minimize(f0_barrier, x0=[0.0, 0.0], args=(0.01,), method="Nelder-Mead")
print("solution barrière :", sol_barrier.x, "f0 =", f0(*sol_barrier.x))


**(b) SLSQP avec contraintes explicites.** Cette fois, on passe directement les 4 contraintes $g_i(x)\ge 0$ à `scipy.optimize.minimize(..., method="SLSQP", constraints=...)`, sans les transformer en pénalité.

In [ ]:
# A COMPLETER : cons = tuple de 4 dicts {"type": "ineq", "fun": lambda x: g_i(x)}
cons = None
sol_slsqp = minimize(lambda x: f0(x[0], x[1]), x0=[0.0, 0.0], method="SLSQP", constraints=cons)
print("solution SLSQP    :", sol_slsqp.x, "f0 =", f0(*sol_slsqp.x))


**(c) `cvxpy`.** Le problème est convexe : formulez-le directement en `cvxpy` (variables, contrainte `cp.abs(x1) + cp.abs(x2) <= 1`, objectif, `prob.solve()`).

In [ ]:
# A COMPLETER : x1, x2 = cp.Variable() (x2) ; constraints = [...] ; obj = cp.Minimize(...) ; prob = cp.Problem(obj, constraints) ; prob.solve()
raise NotImplementedError("Complétez la formulation cvxpy")
print("solution cvxpy    :", x1.value, x2.value, "f0 =", prob.value)


**✅ Vérification 3.1** — les trois méthodes doivent tomber sur (à peu près) le même point.

In [ ]:
_pts = {
    "barrière": np.array(sol_barrier.x),
    "SLSQP": np.array(sol_slsqp.x),
    "cvxpy": np.array([x1.value, x2.value]),
}
for name, p in _pts.items():
    print(f"  {name:10s}: {np.round(p, 4)}")
_ref = _pts["SLSQP"]
for name, p in _pts.items():
    assert np.linalg.norm(p - _ref) < 0.05, f"{name} s'écarte trop de SLSQP : {p} vs {_ref}"
print("OK : les trois méthodes s'accordent (à 0.05 près).")


**3.2 — Multiplicateurs de Lagrange à la main.** Écrivez les quatre contraintes sous la forme $g_i(x) \ge 0$ :
$$ g_1 = 1-x_1-x_2,\quad g_2 = 1+x_1+x_2,\quad g_3 = 1-x_1+x_2,\quad g_4 = 1+x_1-x_2 . $$
Identifiez numériquement quelles $g_i$ sont actives (nulles) à l'optimum — vous devriez trouver **deux** contraintes actives simultanément, car l'optimum se situe exactement sur un *sommet* du losange $|x_1|+|x_2|\le 1$ (le même phénomène qu'un sommet optimal du polytope d'un PL !). Sur une feuille, écrivez la condition de stationnarité KKT avec les deux multiplicateurs correspondants,
$$ \nabla f_0(x^\star) = \mu_1 \nabla g_1(x^\star) + \mu_3 \nabla g_3(x^\star), \qquad \mu_1,\mu_3 \ge 0, $$
résolvez le système linéaire 2×2 obtenu **à la main**, puis codez le même calcul ci-dessous pour vérifier votre résultat papier.

In [ ]:
x_star = np.array(sol_slsqp.x)
g = [1 - x_star[0] - x_star[1], 1 + x_star[0] + x_star[1],
     1 - x_star[0] + x_star[1], 1 + x_star[0] - x_star[1]]
print("valeurs des contraintes g1..g4 :", np.round(g, 4), " (0 = active)")

# A COMPLETER :
# grad_f0 = gradient de f0 en x_star : (2*(x1-1.5), 4*(x2-0.5)**3)
# M = matrice 2x2 dont les COLONNES sont grad(g1)=(-1,-1) et grad(g3)=(-1,1)
# mu = solution de M @ mu = grad_f0  (np.linalg.solve)
grad_f0 = None
M = None
mu = None
print("mu1, mu3 =", np.round(mu, 4), " (les deux doivent être >= 0 : complémentarité vérifiée)")


**✅ Vérification 3.2** — la stationnarité KKT doit être satisfaite (résidu quasi nul) et les deux multiplicateurs doivent être positifs.

In [ ]:
_grad_g1, _grad_g3 = np.array([-1, -1]), np.array([-1, 1])
_residual = grad_f0 - (mu[0] * _grad_g1 + mu[1] * _grad_g3)
assert np.linalg.norm(_residual) < 1e-4, f"stationnarité KKT non vérifiée, résidu = {_residual}"
assert mu[0] > -1e-6 and mu[1] > -1e-6, f"les deux multiplicateurs doivent être >= 0, obtenu {mu}"
print(f"OK : résidu KKT = {np.linalg.norm(_residual):.2e}, mu = {np.round(mu, 4)} (>= 0)")


## Exercice 4 — Programmation linéaire : mix de production

Avant l'extension de sa gamme avec le modèle **Sport** (voir le Projet, Partie 1 ci-dessous, pour la gamme réelle à trois produits), ExedBike ne fabriquait que deux modèles de vélos électriques : **Urban** (marge 120€/unité) et **Cargo** (marge 200€/unité) — c'est cette version simplifiée, déjà résolue à la main en TD1 (Partie ExedBike — Résolution), que nous reprenons ici pour la vérifier avec un solveur.
Chaque semaine, l'usine dispose de **480 heures de main-d'œuvre**, **900 kg d'aluminium pour batteries**, et **240 heures-machine**.

| Ressource            | Urban | Cargo | Disponible |
|---------------------|:----:|:----:|:---:|
| Main-d'œuvre (h/unité)      | 4    | 6    | 480 |
| Aluminium (kg/unité)  | 6    | 15    | 900 |
| Temps machine (h/unité)| 2    | 2    | 240 |

**4.1a** Formulez et résolvez le PL avec `scipy.optimize.linprog` (rappel : `linprog` **minimise** — pour maximiser la marge, minimisez son opposé).

In [ ]:
from scipy.optimize import linprog

# A COMPLETER :
# c      = coefficients de l'objectif A MINIMISER (donc l'opposé des marges 120, 200)
# A_ub   = matrice des 3 contraintes <=  (main-d'oeuvre, aluminium, temps machine)
# b_ub   = second membre des 3 contraintes [480, 900, 240]
c = None
A_ub = None
b_ub = None

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=(0, None), method="highs")
print("scipy.optimize.linprog :")
print("  Urban =", round(res.x[0], 2), " Cargo =", round(res.x[1], 2),
      " marge hebdomadaire = ", round(-res.fun, 2), "€")


**4.1b** Formulez le **même** PL avec `PuLP` (langage plus proche de l'écriture mathématique : variables, objectif à *maximiser* directement, contraintes une par une).

In [ ]:
import pulp

prob = pulp.LpProblem("ExedBike_mix", pulp.LpMaximize)
# A COMPLETER :
# u, cg  = deux LpVariable (Urban, Cargo), bornées >= 0 (lowBound=0)
# prob  += l'objectif 120*u + 200*cg
# prob  += les 3 contraintes de ressources (<= 480, <= 900, <= 240), avec un nom pour chacune
u = None
cg = None

prob.solve(pulp.PULP_CBC_CMD(msg=False))
print("Statut PuLP :", pulp.LpStatus[prob.status])
print("Urban =", u.value(), " Cargo =", cg.value(), " marge = ", pulp.value(prob.objective), "€")

for name, con in prob.constraints.items():
    print(f"  {name:14s} marge de manoeuvre = {con.slack:.2f}  (saturée : {abs(con.slack) < 1e-6})")


**✅ Vérification 4.1** — `scipy` et `PuLP` doivent donner exactement la même marge optimale (et la même que votre tableau de simplexe à la main en TD1 : 15 000 €).

In [ ]:
_obj_scipy = -res.fun
_obj_pulp = pulp.value(prob.objective)
assert abs(_obj_scipy - _obj_pulp) < 1e-2, f"scipy ({_obj_scipy}) et PuLP ({_obj_pulp}) ne s'accordent pas"
assert abs(_obj_scipy - 15000) < 1.0, f"attendu 15 000 € (cf. TD1), obtenu {_obj_scipy}"
print(f"OK : scipy = {_obj_scipy:.2f} € ; PuLP = {_obj_pulp:.2f} € ; TD1 (à la main) = 15000 €")


**4.2** Quelles contraintes sont saturées à l'optimum ? Correspondent-elles au sommet identifié à la main en TD ? Sur quelle ressource faudrait-il investir en priorité ? Affichez les prix duaux (`prob.constraints[name].pi` en PuLP, `res.ineqlin.marginals` en `linprog`) et interprétez-les.

In [ ]:
# A COMPLETER : affichez les prix duaux de chaque contrainte, pour PuLP (con.pi) et pour scipy (res.ineqlin.marginals)
print("Prix duaux (PuLP) :")

print("\nValeurs duales (scipy) :")


---
## Projet individuel ExedBike — Partie 1 (à rendre avant la Séance 2)

Vous allez travailler sur ce **projet fil rouge** tout au long des trois séances. Lisez le brief complet du projet (document séparé) pour le contexte, les jalons et les critères d'évaluation. Voici la **Partie 1** — contrairement aux exercices précédents, **c'est votre travail individuel noté** : il n'y a pas de cellule de vérification qui vous donne la réponse.

### 1. Ajuster un modèle de coût de production (optimisation continue)

Les données ci-dessous forment un jeu de données synthétique du coût unitaire de production en fonction du volume cumulé produit pour le modèle Urban (un effet classique de **courbe d'apprentissage** : le coût baisse avec l'expérience accumulée). Le modèle est une loi de puissance,
$$ c(q) = k\, q^{-\theta}, \qquad \theta \in (0,1). $$
En passant au logarithme, ceci devient une **régression linéaire** : $\log c(q) = \log k - \theta \log q$.

In [ ]:
# Jeu de données synthétique (ne changez pas la graine : les résultats doivent être reproductibles)
rng = np.random.default_rng(7)
q = np.linspace(50, 2000, 40)
k_true, theta_true = 9500.0, 0.55
c_obs = k_true * q ** (-theta_true) * np.exp(rng.normal(0, 0.04, size=q.shape))

plt.figure()
plt.scatter(q, c_obs, s=15, label="coût unitaire observé")
plt.xlabel("volume cumulé (unités)")
plt.ylabel("coût unitaire (EUR)")
plt.title("ExedBike Urban — données de courbe d'apprentissage")
plt.legend()
plt.show()


Ajustez $(\log k, \theta)$ en minimisant l'erreur quadratique moyenne **avec votre propre implémentation de la descente de gradient** (celle de l'Exercice 1, réutilisable telle quelle) sur les données log-log **standardisées**, puis reportez $(k,\theta)$ dans les unités d'origine. Tracez les données et l'ajustement, ainsi que la courbe de convergence, et expliquez (dans votre synthèse d'une page) pourquoi standardiser $\log q$ compte ici (faites le lien avec le mauvais conditionnement observé à l'Exercice 1.2).

Étapes suggérées :
1. `x = log(q)`, `y = log(c_obs)` ;
2. standardisez `x` : `x_z = (x - x.mean()) / x.std()` ;
3. définissez `J_fit(params)` = EQM entre `p0 - p1*x_z` et `y` (avec `params=[p0,p1]`, en utilisant `anp` pour que `autograd` puisse dériver) ;
4. `gradient_descent(J_fit, w_init=[0.0, 0.0], alpha=..., epochs=...)` ;
5. dé-standardisez : `theta_hat = p1/x_std`, `k_hat = exp(p0 + p1*x_mean/x_std)` ;
6. tracez le nuage de points + la courbe $k\_hat \cdot q^{-\theta\_hat}$, et la courbe de convergence de la perte.

In [ ]:
# A COMPLETER (suivez les 6 étapes ci-dessus)
x = np.log(q)
y = np.log(c_obs)


# ... standardisation, J_fit, gradient_descent, dé-standardisation ...


# print(f"convergé en {len(hist)-1} itérations")
# print(f"ajustement : k = {k_hat:.1f}, theta = {theta_hat:.3f}")

# ... graphiques (nuage + ajustement, courbe de convergence) ...


### 2. PL du mix de production pour la gamme réelle

ExedBike vend en réalité **trois** modèles : Urban, Cargo et Sport, avec les données ci-dessous. Formulez et résolvez le PL du mix de production hebdomadaire (maximiser la marge sous les contraintes de ressources) avec `PuLP` **ou** `scipy.optimize.linprog` — vous savez déjà faire les deux depuis l'Exercice 4, c'est la même démarche avec un troisième produit. Reportez le mix optimal, la marge, les contraintes saturées et les prix duaux, et donnez une phrase d'interprétation métier pour chaque prix dual.

| Ressource | Urban | Cargo | Sport | Disponible |
|---|:--:|:--:|:--:|:--:|
| Main-d'œuvre (h/unité) | 4 | 6 | 5 | 520 |
| Aluminium (kg/unité) | 6 | 15 | 9 | 1100 |
| Temps machine (h/unité) | 2 | 2 | 3 | 260 |
| Marge (€/unité) | 120 | 200 | 175 | — |

In [ ]:
# A COMPLETER — Partie 1, Question 2 (à rendre) :
# 1) Définissez vos 3 variables de décision (Urban, Cargo, Sport)
# 2) Écrivez l'objectif : maximiser la marge totale
# 3) Écrivez les 3 contraintes de ressources (table ci-dessus)
# 4) Résolvez avec scipy.optimize.linprog OU PuLP (au choix -- un seul suffit ici)
# 5) Affichez : mix optimal, marge totale, contraintes saturées, prix duaux


### Livrables pour la Partie 1 (à rendre sous forme d'un seul notebook + une synthèse PDF d'une page)
- Ce notebook complété : Exercices 1 à 4 (toutes les cellules `# A COMPLETER`, vérifications au vert) et les deux tâches du projet ci-dessus.
- Une courte synthèse (≤1 page) : paramètres du modèle de coût ajusté, mix de production optimal, et un paragraphe interprétant les prix duaux pour le directeur d'usine (langage non technique).

Ce travail sera prolongé en Séances 2 et 3 (décisions sur les lignes de production, ordonnancement, réseau de distribution).